# Session Control — Claude Managed Agents (CLI)

The same controls as [`05-session-control/`](../05-session-control/), with the **`ant`** CLI:

| Control | Command |
|---|---|
| **Cap the spend** | `ant beta:sessions create --budget ...`; watch `session.usage` on the stream |
| **Add context mid-session** | send a `system.message` |
| **Change direction** | send `user.interrupt` + a new `user.message` |
| **Survive a dropped connection** | reopen `ant ... stream`, catch up with `ant beta:sessions:events list` |
| **Audit what happened** | `ant beta:sessions:events list --type ...` |
| **Pause at the cap, then resume** | `stop_reason: budget_reached`; `ant beta:sessions update --budget ...` |
| **Inspect and close** | `ant beta:sessions retrieve / list / archive` |

> Prerequisite: the `ant` CLI (>= 1.40) installed and authenticated — see [`01-basics-cli/`](../01-basics-cli/).

## 1. Setup

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

# `ant` is a separate process from this notebook, so a bit of setup:
#  - it doesn't read the root .env — load the API key into os.environ, which
#    every `!ant` call below inherits;
#  - Jupyter sets FORCE_COLOR=1, which makes ant wrap its JSON output in
#    ANSI color codes and break parsing — clear it so the output stays plain;
#  - `!` runs commands in a pseudo-terminal, so ant probes it with escape
#    sequences that leak into the cell output — TERM=dumb turns that off.
load_dotenv(find_dotenv())
os.environ.pop("FORCE_COLOR", None)
os.environ.pop("CLICOLOR_FORCE", None)
os.environ["NO_COLOR"] = "1"
os.environ["TERM"] = "dumb"
assert os.environ.get("ANTHROPIC_API_KEY"), (
    "ANTHROPIC_API_KEY not found — add it to the root .env or run `ant auth login`."
)
print("Credential loaded, color disabled — ant is ready")

In [ ]:
%%writefile environment.yaml
# yaml-language-server: $schema=https://platform.claude.com/schemas/ant/beta/environment.json
name: session-control-env-cli
config:
  type: cloud
  networking:
    type: unrestricted

In [ ]:
%%writefile agent.md
---
name: Session Control Agent (CLI)
model: claude-opus-5-5
tools:
  - type: agent_toolset_20260401
---

You are a helpful engineer working in a Linux sandbox. Keep replies short.

In [ ]:
import json

!ant apply environment.yaml agent.md --yes --lock-file claude-lock.json

lock = json.load(open("claude-lock.json"))["resources"]
env_id = lock["./environment.yaml"]["id"]
agent_id, agent_version = lock["./agent.md"]["id"], int(lock["./agent.md"]["version"])
print(f"Environment : {env_id}")
print(f"Agent       : {agent_id}  (v{agent_version})")

Helpers. One-shot commands that return data run through `ant_json()`: it captures `--format json` output, so there's no terminal and no color codes. The streaming pieces, `send()` and `events()`, are the same as in modules 03 and 04.

In [ ]:
import subprocess
import time


def ant(*args: str, input: str | None = None) -> str:
    """Run an ant command and return its stdout; show stderr if it fails."""
    result = subprocess.run(["ant", *args], input=input, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip())
    return result.stdout


def ant_json(*args: str):
    return json.loads(ant(*args, "--format", "json"))


def ant_jsonl(*args: str) -> list:
    return [json.loads(line) for line in ant(*args, "--format", "jsonl").splitlines() if line.strip()]


def send(session_id: str, *events: dict) -> list:
    """Send events; returns the queued events (with their ids)."""
    return json.loads(ant("beta:sessions:events", "send", "--session-id", session_id, "--format", "json",
                          input=json.dumps({"events": list(events)})))["data"]


def open_stream(session_id: str):
    return subprocess.Popen(
        ["ant", "beta:sessions:events", "stream", "--session-id", session_id, "--format", "json"],
        stdout=subprocess.PIPE, text=True, bufsize=1,
    )


def events(stream):
    """Yield each event from `ant ... stream --format json` as a dict."""
    buf = []
    for line in stream.stdout:
        buf.append(line)
        if line.rstrip("\n") == "}":
            yield json.loads("".join(buf))
            buf = []


def message(text: str) -> dict:
    return {"type": "user.message", "content": [{"type": "text", "text": text}]}


def text_of(event: dict) -> str:
    return "".join(b["text"] for b in event.get("content", []) if b.get("type") == "text")


def cents(amount: dict) -> str:
    return f"${int(amount['amount']) / 100:.2f}"

## 2. Create a Session with a Budget

`--budget` takes the same object as the API. `amount` is a whole number of **US cents as a string** — `"200"` is $2.00 — and the budget can only be attached **at creation**. List cost covers model tokens, web searches ($10 / 1,000), and running time ($0.08 / hour). At the cap the session **pauses**; it doesn't terminate.

In [ ]:
agent_ref = json.dumps({"type": "agent", "id": agent_id, "version": agent_version})
budget = json.dumps({"type": "limit", "max_list_cost": {"amount": "200", "currency": "USD"}})

_session = !ant beta:sessions create --agent '{agent_ref}' --environment-id {env_id} --title "Session control CLI" --budget '{budget}' --transform id -r  # type: ignore
session_id = _session[0]

s = ant_json("beta:sessions", "retrieve", "--session-id", session_id)
print(f"Session ID : {session_id}  [{s['status']}]  budget {cents(s['budget']['max_list_cost'])}")

## 3. A `follow()` Helper

It reads an **already-open** stream until the turn ends and returns the `stop_reason`. New events it prints:

| Event | Meaning |
|---|---|
| `user.message` | Echoed when the agent picks it up; queued messages have `processed_at: null` until then |
| `user.interrupt` | Your interrupt was applied |
| `session.usage` | Cumulative usage and list cost, emitted right before every idle (echoes the budget) |
| `session.error` | Something went wrong; carries a typed `error` |

`after_text` handles interrupts (section 5).

In [ ]:
def follow(stream, *, after_text=None, on_event=None) -> str:
    """Print events until the turn ends; return the stop_reason type."""
    waiting_for = after_text
    for event in events(stream):
        if on_event:
            on_event(event)
        match event["type"]:
            case "user.message":
                print(f"[user] {text_of(event)[:80]}")
                if text_of(event) == waiting_for:
                    waiting_for = None
            case "user.interrupt":
                print("[interrupt applied]")
            case "agent.message":
                print(f"Agent: {text_of(event)}")
            case "agent.tool_use":
                print(f"[{event['name']}] {str(event['input'].get('command') or event['input'].get('file_path') or '')[:90]}")
            case "session.usage":
                b = f" of {cents(event['budget']['max_list_cost'])}" if event.get("budget") else ""
                print(f"[usage] {cents(event['usage']['list_cost'])}{b}, {event['usage']['active_seconds']:.0f}s active")
            case "session.error":
                print(f"[error] {event.get('error')}")
            case "session.status_idle":
                reason = event["stop_reason"]["type"]
                if reason == "requires_action":
                    continue
                if reason == "end_turn" and waiting_for:
                    print("[idle: end_turn — the interrupted turn; waiting for the redirect]")
                    continue
                print(f"[idle: {reason}]")
                return reason
            case "session.status_terminated":
                print("[terminated]")
                return "terminated"

## 4. Add Context Mid-Session with `system.message`

`system.message` appends privileged, system-level context for this turn and every later one, without replacing the agent's system prompt. Send it in the same request as the `user.message` it informs, **as the last event** — otherwise the request is rejected with a 400 error.

In [ ]:
stream = open_stream(session_id)                      # stream first
send(
    session_id,
    message("What's my local time right now? Check the sandbox clock with bash."),
    {"type": "system.message", "content": [{"type": "text", "text": "The user is in Madrid (Europe/Madrid timezone)."}]},
)
follow(stream)
stream.terminate()

## 5. Interrupt and Redirect

Send `user.interrupt` and a new `user.message` in one request. The interrupted turn ends with a **`session.status_idle` whose `stop_reason` is `end_turn`** — indistinguishable from a normal finish — and then the redirect runs as a new turn. So after an interrupt, only count an `end_turn` that comes after your redirect is processed: `follow(after_text=...)`.

From the shell, the same pair of events is one command:

```bash
ant beta:sessions:events send --session-id "$SID" <<'YAML'
events:
  - type: user.interrupt
  - type: user.message
    content:
      - type: text
        text: Stop that. Instead, ...
YAML
```

In [ ]:
REDIRECT = "Stop that. Instead, just tell me the Python version installed in the sandbox."
interrupted = False


def interrupt_on_first_tool(event: dict) -> None:
    global interrupted
    if event["type"] == "agent.tool_use" and not interrupted:
        interrupted = True
        send(session_id, {"type": "user.interrupt"}, message(REDIRECT))
        print(">>> sent user.interrupt + redirect")


stream = open_stream(session_id)
send(session_id, message(
    "Use bash to compute the SHA-256 hash of each of the strings 'a' through 'j', "
    "one separate command per string, and report all ten."
))
follow(stream, after_text=REDIRECT, on_event=interrupt_on_first_tool)
stream.terminate()

The tool calls the agent had requested never ran, the interrupted turn's `end_turn` was skipped, and the redirect ran as a fresh turn.

> A `user.message` sent while the agent is running is **queued**, not applied mid-turn. Interrupt first when the agent must stop now.

## 6. Reconnect Without Missing Events

A new stream only delivers events emitted after it opens. To reconnect: **open the stream**, **list the history** and remember the ids you've seen, then **tail the stream**, skipping those ids. We simulate a dropped client by sending with no stream open and waiting 5 seconds.

In [ ]:
sent = send(session_id, message("Use bash to count the entries in /etc, then tell me the count."))
my_message_id = sent[0]["id"]
print(f"Sent {my_message_id} with no stream open — 'disconnected' for 5 seconds...")
time.sleep(5)

In [ ]:
def print_event(event: dict) -> None:
    if event["type"] == "agent.message":
        print(f"Agent: {text_of(event)}")
    elif event["type"] == "agent.tool_use":
        print(f"[{event['name']}] {str(event['input'].get('command') or '')[:90]}")


stream = open_stream(session_id)                                              # 1. stream first
history = ant_jsonl("beta:sessions:events", "list", "--session-id", session_id)  # 2. catch up
seen = {event["id"] for event in history}

start = next((i for i, e in enumerate(history) if e["id"] == my_message_id), len(history))
missed = history[start + 1:]
print(f"--- caught up {len(missed)} events from history ---")
for event in missed:
    print_event(event)

finished = any(e["type"] == "session.status_idle" and e["stop_reason"]["type"] == "end_turn" for e in missed)
if finished:
    print("--- the turn finished while we were away ---")
else:
    print("--- now tailing the live stream ---")
    for event in events(stream):                                              # 3. tail, de-duplicated
        if event["id"] in seen:
            continue
        seen.add(event["id"])
        print_event(event)
        if event["type"] == "session.status_idle" and event["stop_reason"]["type"] != "requires_action":
            print(f"[idle: {event['stop_reason']['type']}]")
            break
stream.terminate()

## 7. Audit with the Event History

`ant beta:sessions:events list` takes `--type` (repeatable) to filter. Every command the agent ran in this session:

In [ ]:
for event in ant_jsonl("beta:sessions:events", "list", "--session-id", session_id, "--type", "agent.tool_use"):
    print(f"{event['processed_at'][11:19]}  [{event['name']}] {str(event['input'].get('command') or '')[:80]}")

## 8. Pause at the Budget, Then Resume

Lower the cap to just above what the session has consumed (a new cap must be **strictly greater**), give it a multi-step task, and it pauses with **`stop_reason: budget_reached`**. The request that crosses the cap still finishes, so the cost can land a little past it.

While paused, the session rejects new `user.message` events. **`ant beta:sessions update --budget ...`** raises the cap and the paused work **resumes on its own** — so we keep the same stream open across the update.

In [ ]:
def set_budget(amount_cents: int) -> None:
    ant("beta:sessions", "update", "--session-id", session_id,
        "--budget", json.dumps({"type": "limit", "max_list_cost": {"amount": str(amount_cents), "currency": "USD"}}))


consumed = int(ant_json("beta:sessions", "retrieve", "--session-id", session_id)["usage"]["list_cost"]["amount"])
set_budget(consumed + 1)
print(f"Consumed so far: {consumed} cents -> budget lowered to {consumed + 1} cents")

In [ ]:
TASK = (
    "Work in small steps, one bash command per step, and look at each result before the next step: "
    "1) create /tmp/lab, 2) write a Python script there that prints the first 20 primes, 3) run it, "
    "4) write a second script that prints the first 20 Fibonacci numbers, 5) run it, "
    "6) list /tmp/lab. Then summarize."
)

stream = open_stream(session_id)
send(session_id, message(TASK))
reason = follow(stream)

if reason == "budget_reached":
    paused = ant_json("beta:sessions", "retrieve", "--session-id", session_id)
    print(f"\n>>> paused at {cents(paused['usage']['list_cost'])} — raising the budget by $1.00\n")
    set_budget(int(paused["usage"]["list_cost"]["amount"]) + 100)
    follow(stream)   # same stream: the paused work resumes by itself
stream.terminate()

If the request that crossed the cap also finished the agent's work, raising the budget finds nothing left to do; usually it pauses earlier and resumes mid-task. In a multiagent session (modules 06–07), **one budget covers every agent** and pauses workflow runs too.

## 9. Inspect, List, Archive

- `ant beta:sessions retrieve` — `status` and cumulative `usage`
- `ant beta:sessions list` — page through sessions
- `ant beta:sessions archive` — read-only from now on; history kept. Interrupt a `running` session and wait for `idle` first.
- `ant beta:sessions delete` — removes the session, its events, sandbox, **and its output files**

To watch or steer a session by hand, `ant beta:sessions connect <session-id>` attaches an interactive terminal.

In [ ]:
s = ant_json("beta:sessions", "retrieve", "--session-id", session_id)
u = s["usage"]
print(f"Status      : {s['status']}")
print(f"List cost   : {cents(u['list_cost'])}  (budget {cents(s['budget']['max_list_cost'])})")
print(f"Tokens      : {u['input_tokens']} in, {u['output_tokens']} out, {u['cache_read_input_tokens']} cache reads")
print(f"Active time : {u['active_seconds']:.0f}s")

print("\nRecent sessions:")
for row in ant_jsonl("beta:sessions", "list", "--limit", "3", "--max-items", "3", "--transform", "{id,status,title}"):
    print(f"  {row['id']}  {row['status']:<10} {row['title']}")

In [ ]:
!ant beta:sessions archive --session-id {session_id} --transform archived_at -r

## Summary

```
ant beta:sessions create ... --budget '{"type":"limit","max_list_cost":{"amount":"200","currency":"USD"}}'
session.usage                     →  cumulative list cost before every idle
send [..., system.message]        →  system context mid-session (last event in the request)
send [user.interrupt, user.message]  →  stop + redirect; skip the interrupted turn's end_turn
reconnect                         →  stream → ant beta:sessions:events list → tail, skipping seen ids
ant beta:sessions:events list --type agent.tool_use   →  audit
budget_reached                    →  ant beta:sessions update --budget ... resumes the work
ant beta:sessions retrieve / list / archive / delete / connect
```

### CLI vs SDK, same lesson

| | CLI (this module) | SDK ([`05-session-control/`](../05-session-control/)) |
|---|---|---|
| Budget at creation | `--budget '{...}'` | `budget={...}` |
| Change the budget | `ant beta:sessions update --budget` | `client.beta.sessions.update(budget=...)` |
| History | `ant beta:sessions:events list [--type ...]` | `client.beta.sessions.events.list(types=[...])` |
| Follow by hand | `ant beta:sessions connect` | your own UI |

### Next steps
- **06-subagents** — delegate to other agents in session threads
- **07-dynamic-workflows** — let the agent write a workflow that runs many agents in parallel